# Importación de librerias necesarias

In [34]:
import pandas as pd
import numpy as np

# Obtención y preparación de los datos


In [ ]:
df = pd.read_csv('/workspace/dataset.csv')

# Análisis exploratorio de datos (EDA)

Nuestro primer paso será mostrar los primeros 5 registros del dataset para tener una idea de cómo se ve:

In [37]:
# Primeros 5 registros del dataset
df.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Tras esto vamos a ver qué tipo de dato es cada columna del DataFrame y veremos que columnas contienen valores faltantes:

In [11]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

In [12]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Como se puede ver efectivamente hay columnas que contienen valores nulos. Concretamente Age, Cabin y Embarked.

Por último, vamos a obtener un resumen de la información del DataFrame:

In [13]:
# Información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


# Limpieza de valores faltantes

Primero nos vamos a centrar en las columnas numéricas:

In [38]:
df.select_dtypes(include=np.number).isna().sum()

PassengerId      0
Survived         0
Pclass           0
Age            177
SibSp            0
Parch            0
Fare             0
dtype: int64

Que como vimos en el anterior apartado se trata de la columna Age. En este caso creemos conveniente usar la mediana para rellenar los valores faltantes pues es mucho más robusta frente a valores atípicos que la media.

In [39]:
mediana_edad = df['Age'].median()
df['Age'] = df['Age'].fillna(mediana_edad)

Ahora pasamos a las columnas de tipo texto que como vimos anteriormente son tanto Embarked como Cabin. Usaremos estrategias diferentes:

Para el caso de Cabin falta la mayoría de datos. Emplear la moda aquí no tiene sentido y se ve mejor aplicar una cadena de texto específica como puede ser "Desconocido"

In [40]:
df['Cabin'] = df['Cabin'].fillna('Desconocido')

Por otro lado tenemos Embarked donde solo faltan dos valores, emplear aquí la moda es bastante útil pues asumimos que se subieron en el puerto donde subió la inmensa mayoría de la gente. 

In [42]:
moda_embarque = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(moda_embarque)

# Transformación de datos 

Ahora convertiremos las variables categóricas a formato numérico utilizando one-hot encoding para facilitar el análisis convirtiendo variables categóricas en columnas binarias (dummies), lo cual es necesario para futuros modelados.

In [43]:
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

En este dataset será necesario normalizar dos variables numéricas pues tienen escalas diferentes como son Age y Fare. Para escalarlas usaremos la normalización min-max.

In [44]:
df['Age'] = (df['Age'] - df['Age'].min()) / (df['Age'].max() - df['Age'].min())

df['Fare'] = (df['Fare'] - df['Fare'].min()) / (df['Fare'].max() - df['Fare'].min())



Tras estos cambios vamos a verificar que los datos estén listos. Para ello veremos que las columnas de texto han cambiado:

In [45]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    object 
 8   Fare         891 non-null    float64
 9   Cabin        891 non-null    object 
 10  Sex_male     891 non-null    bool   
 11  Embarked_Q   891 non-null    bool   
 12  Embarked_S   891 non-null    bool   
dtypes: bool(3), float64(2), int64(5), object(3)
memory usage: 72.3+ KB
None


Y por último veremos los 5 primeros registros para ver como quedarían estos cambios:


In [46]:
df.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Sex_male,Embarked_Q,Embarked_S
0,1,0,3,"Braund, Mr. Owen Harris",0.271174,1,0,A/5 21171,0.014151,Desconocido,True,False,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0.472229,1,0,PC 17599,0.139136,C85,False,False,False
2,3,1,3,"Heikkinen, Miss. Laina",0.321438,0,0,STON/O2. 3101282,0.015469,Desconocido,False,False,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0.434531,1,0,113803,0.103644,C123,False,False,True
4,5,0,3,"Allen, Mr. William Henry",0.434531,0,0,373450,0.015713,Desconocido,True,False,True



Antes de guardar la versión limpia de los datos, el último paso para tener los datos preparados de manera efectiva para un análisis o modelado posterior es eliminar las otras columnas que quedan de texto:

In [47]:
df.drop(columns=['Name', 'Ticket', 'Cabin'], inplace=True)

Ahora sí procedemos a guardar el dataset limpio:

In [ ]:
ruta_guardado = '/workspace/dataset_cleaned.csv'
df.to_csv(ruta_guardado, index=False)

print(f"Dataset limpio y transformado guardado exitosamente en: {ruta_guardado}")

Dataset limpio y transformado guardado exitosamente en: ./dataset_cleaned.csv
